In [4]:
import pandas as pd

df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')
print(df.shape)
df.head()

(253680, 22)


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [6]:
import pandas as pd

df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')
print(df.shape)
df.head()

print("Columns:")
print(df.columns.tolist())
print()

print("Missing values per column:")
print(df.isnull().sum().sum(), "total missing values")
print()

print("Class distribution (Diabetes_binary):")
print(df['Diabetes_binary'].value_counts())
print()
print("Class proportions:")
print(df['Diabetes_binary'].value_counts(normalize=True))


(253680, 22)
Columns:
['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']

Missing values per column:
0 total missing values

Class distribution (Diabetes_binary):
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64

Class proportions:
Diabetes_binary
0.0    0.860667
1.0    0.139333
Name: proportion, dtype: float64


In [8]:
y = df['Diabetes_binary']

lifestyle_features = [
    'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke',
    'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
    'HvyAlcoholConsump', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk'
]

socioeconomic_features = [
    'AnyHealthcare', 'NoDocbcCost', 'Sex', 'Age', 'Education', 'Income'
]

all_features = lifestyle_features + socioeconomic_features

print(f"Lifestyle/health features ({len(lifestyle_features)}):", lifestyle_features)
print()
print(f"Socioeconomic features ({len(socioeconomic_features)}):", socioeconomic_features)
print()
print(f"Total features: {len(all_features)}")

Lifestyle/health features (15): ['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk']

Socioeconomic features (6): ['AnyHealthcare', 'NoDocbcCost', 'Sex', 'Age', 'Education', 'Income']

Total features: 21


In [19]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df_sample = df.sample(n=50000, random_state=42)
print("Sampled shape:", df_sample.shape)

y_sample = df_sample['Diabetes_binary']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
dt = Pipeline([('scaler', StandardScaler()), ('clf', DecisionTreeClassifier(random_state=42, class_weight='balanced'))])
rf = Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1))])
knn = Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=5, n_jobs=-1))])

models = {'Logistic Regression': lr, 'Decision Tree': dt, 'Random Forest': rf, 'K-NN': knn}

scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

print("Models set up:", list(models.keys()))

Sampled shape: (50000, 22)
Models set up: ['Logistic Regression', 'Decision Tree', 'Random Forest', 'K-NN']


In [23]:
import pandas as pd

X_all = df_sample[all_features]

results_all = {}

for name, model in models.items():
    print(f"Training {name}...")
    cv_results = cross_validate(model, X_all, y_sample, cv=cv, scoring=scoring, n_jobs=-1)
    results_all[name] = cv_results
    print(f"  Done. F1 = {cv_results['test_f1'].mean():.3f}, ROC-AUC = {cv_results['test_roc_auc'].mean():.3f}")

print("\nAll models trained!")

summary_all = pd.DataFrame({
    name: {
        'Accuracy': res['test_accuracy'].mean(),
        'Precision': res['test_precision'].mean(),
        'Recall': res['test_recall'].mean(),
        'F1': res['test_f1'].mean(),
        'ROC-AUC': res['test_roc_auc'].mean(),
    }
    for name, res in results_all.items()
}).T

print("\nResults on combined features (lifestyle + socioeconomic):")
print(summary_all.round(3))


Training Logistic Regression...
  Done. F1 = 0.445, ROC-AUC = 0.826
Training Decision Tree...
  Done. F1 = 0.289, ROC-AUC = 0.588
Training Random Forest...
  Done. F1 = 0.214, ROC-AUC = 0.806
Training K-NN...
  Done. F1 = 0.266, ROC-AUC = 0.717

All models trained!

Results on combined features (lifestyle + socioeconomic):
                     Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression     0.733      0.312   0.773  0.445    0.826
Decision Tree           0.804      0.290   0.289  0.289    0.588
Random Forest           0.862      0.496   0.136  0.214    0.806
K-NN                    0.848      0.397   0.200  0.266    0.717


In [27]:
feature_sets = {'Lifestyle Only': lifestyle_features, 'Socioeconomic Only': socioeconomic_features, 'Combined': all_features}

comparison_results = []

for set_name, features in feature_sets.items():
    print(f"\n=== Feature set: {set_name} ({len(features)} features) ===")
    X_subset = df_sample[features]
    for model_name, model in models.items():
        cv_results = cross_validate(model, X_subset, y_sample, cv=cv, scoring=scoring, n_jobs=-1)
        comparison_results.append({'Feature Set': set_name, 'Model': model_name, 'F1': cv_results['test_f1'].mean(), 'ROC-AUC': cv_results['test_roc_auc'].mean(), 'Recall': cv_results['test_recall'].mean(), 'Precision': cv_results['test_precision'].mean()})
        print(f"  {model_name}: F1={cv_results['test_f1'].mean():.3f}, ROC-AUC={cv_results['test_roc_auc'].mean():.3f}")

comparison_df = pd.DataFrame(comparison_results)
print("\n\n=== Full Comparison ===")
print(comparison_df.round(3).to_string(index=False))




=== Feature set: Lifestyle Only (15 features) ===
  Logistic Regression: F1=0.435, ROC-AUC=0.816
  Decision Tree: F1=0.304, ROC-AUC=0.585
  Random Forest: F1=0.282, ROC-AUC=0.736
  K-NN: F1=0.264, ROC-AUC=0.718

=== Feature set: Socioeconomic Only (6 features) ===
  Logistic Regression: F1=0.338, ROC-AUC=0.703
  Decision Tree: F1=0.324, ROC-AUC=0.675
  Random Forest: F1=0.323, ROC-AUC=0.676
  K-NN: F1=0.096, ROC-AUC=0.604

=== Feature set: Combined (21 features) ===
  Logistic Regression: F1=0.445, ROC-AUC=0.826
  Decision Tree: F1=0.289, ROC-AUC=0.588
  Random Forest: F1=0.214, ROC-AUC=0.806
  K-NN: F1=0.266, ROC-AUC=0.717


=== Full Comparison ===
       Feature Set               Model    F1  ROC-AUC  Recall  Precision
    Lifestyle Only Logistic Regression 0.435    0.816   0.753      0.306
    Lifestyle Only       Decision Tree 0.304    0.585   0.355      0.265
    Lifestyle Only       Random Forest 0.282    0.736   0.254      0.317
    Lifestyle Only                K-NN 0.264    0

In [29]:
import numpy as np

rf_pipeline = models['Random Forest']
rf_pipeline.fit(df_sample[all_features], y_sample)

rf_classifier = rf_pipeline.named_steps['clf']
importances = rf_classifier.feature_importances_

feature_importance_df = pd.DataFrame({'Feature': all_features, 'Importance': importances, 'Type': ['Lifestyle' if f in lifestyle_features else 'Socioeconomic' for f in all_features]}).sort_values('Importance', ascending=False)

print("Feature Importance Ranking (Random Forest):")
print(feature_importance_df.to_string(index=False))

print("\n\nTotal importance by feature type:")
print(feature_importance_df.groupby('Type')['Importance'].sum().round(3))


Feature Importance Ranking (Random Forest):
             Feature  Importance          Type
                 BMI    0.170424     Lifestyle
                 Age    0.127639 Socioeconomic
             GenHlth    0.103542     Lifestyle
              Income    0.084660 Socioeconomic
              HighBP    0.078880     Lifestyle
            PhysHlth    0.065296     Lifestyle
           Education    0.056244 Socioeconomic
            MentHlth    0.051204     Lifestyle
            HighChol    0.045780     Lifestyle
              Fruits    0.028486     Lifestyle
              Smoker    0.027992     Lifestyle
                 Sex    0.027321 Socioeconomic
            DiffWalk    0.025995     Lifestyle
        PhysActivity    0.022513     Lifestyle
             Veggies    0.021620     Lifestyle
HeartDiseaseorAttack    0.018172     Lifestyle
         NoDocbcCost    0.011715 Socioeconomic
   HvyAlcoholConsump    0.009971     Lifestyle
              Stroke    0.009663     Lifestyle
       AnyHealth